# Deploy Custom Deep Research System on OpenShift AI with Kagenti

Deploy the complete document research system to OpenShift using Kagenti Component CRDs.

## 1. Load Environment

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

NAMESPACE = os.getenv("NAMESPACE", "doc-research-lab")
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")

print(f"Namespace: {NAMESPACE}")
print(f"Cluster domain: {CLUSTER_DOMAIN}")
print("✅ Environment loaded")

## 2. Verify Kagenti Installation

Check that the Kagenti operator is installed on the cluster.

In [ ]:
r = subprocess.run(
    ["oc", "get", "crd", "components.kagenti.io"],
    capture_output=True, text=True,
)

if r.returncode == 0:
    print("✅ Kagenti CRDs installed")
    print(r.stdout)
else:
    print("⚠️ Kagenti CRDs not found. Install Kagenti first:")
    print("   See: https://github.com/kagenti/kagenti/blob/main/docs/installation.md")

## 3. Create Namespace and Secrets

In [ ]:
# Create namespace
r = subprocess.run(
    ["oc", "create", "namespace", NAMESPACE, "--dry-run=client", "-o", "yaml"],
    capture_output=True, text=True,
)
if r.returncode == 0:
    apply = subprocess.run(
        ["oc", "apply", "-f", "-"],
        input=r.stdout, capture_output=True, text=True,
    )
    print(apply.stdout.strip())

# Create secret from .env values
secret_path = os.path.join(os.path.dirname(os.getcwd()), "config", "kagenti", "secret.yaml")

# Substitute env vars in the secret template
with open(secret_path) as f:
    secret_yaml = f.read()

for key in ["LLM_BASE_URL", "LLM_API_KEY", "EMBEDDING_BASE_URL", "EMBEDDING_API_KEY",
            "PGVECTOR_USER", "PGVECTOR_PASSWORD"]:
    secret_yaml = secret_yaml.replace(f"${{{key}}}", os.getenv(key, ""))

r = subprocess.run(
    ["oc", "apply", "-f", "-", "-n", NAMESPACE],
    input=secret_yaml, capture_output=True, text=True,
)
print(r.stdout.strip() if r.returncode == 0 else r.stderr)
print("✅ Namespace and secrets configured")

## 4. Deploy PostgreSQL + pgvector

In [ ]:
pg_manifest = """
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: postgresql
spec:
  replicas: 1
  selector:
    matchLabels:
      app: postgresql
  template:
    metadata:
      labels:
        app: postgresql
    spec:
      containers:
      - name: postgres
        image: pgvector/pgvector:pg16
        ports:
        - containerPort: 5432
        env:
        - name: POSTGRES_DB
          value: doc_research
        - name: POSTGRES_USER
          valueFrom:
            secretKeyRef:
              name: doc-research-secret
              key: PGVECTOR_USER
        - name: POSTGRES_PASSWORD
          valueFrom:
            secretKeyRef:
              name: doc-research-secret
              key: PGVECTOR_PASSWORD
        volumeMounts:
        - name: pgdata
          mountPath: /var/lib/postgresql/data
  volumeClaimTemplates:
  - metadata:
      name: pgdata
    spec:
      accessModes: ["ReadWriteOnce"]
      resources:
        requests:
          storage: 10Gi
---
apiVersion: v1
kind: Service
metadata:
  name: postgresql
spec:
  selector:
    app: postgresql
  ports:
  - port: 5432
    targetPort: 5432
"""

r = subprocess.run(
    ["oc", "apply", "-f", "-", "-n", NAMESPACE],
    input=pg_manifest, capture_output=True, text=True,
)
print(r.stdout.strip() if r.returncode == 0 else r.stderr)
print("✅ PostgreSQL+pgvector deployed")

## 5. Deploy Agents via Kagenti Component CRDs

In [ ]:
components_path = os.path.join(
    os.path.dirname(os.getcwd()), "config", "kagenti", "components.yaml"
)

r = subprocess.run(
    ["oc", "apply", "-f", components_path, "-n", NAMESPACE],
    capture_output=True, text=True,
)

if r.returncode == 0:
    print(r.stdout)
    print("✅ Kagenti Components deployed")
else:
    print(f"⚠️ Deployment issue: {r.stderr}")
    print("   Ensure Kagenti operator is installed and images are pushed")

## 6. Verify Deployment

In [ ]:
r = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE, "-o", "wide"],
    capture_output=True, text=True,
)
print("Pods:")
print(r.stdout)

r = subprocess.run(
    ["oc", "get", "components.kagenti.io", "-n", NAMESPACE],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print("\nKagenti Components:")
    print(r.stdout)

print("✅ Deployment verification complete")